# 04 — Anomaly Detection & Recommendations (RQ3, RQ4)
*Can abnormal consumption be detected, and can the system support decisions?*

Hybrid detection (rules + Isolation Forest), episode ranking, KPIs and
quantified recommendation cards.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px

from src.data.ingestion import load_dataset
from src.data.cleaning import clean_dataset

# Default: synthetic data (offline). Switch to source="bdg2" for real BDG2 data.
ds = load_dataset(source="synthetic", n_buildings=8)
print(f"{ds.meters.shape[0]:,} readings | {ds.metadata.shape[0]} buildings")
ds.metadata

In [ ]:
from src.models.anomaly import detect_anomalies, summarize_episodes
from src.recommendations import compute_kpis, generate_recommendations

clean, _ = clean_dataset(ds.meters)
anomalies = detect_anomalies(clean)
episodes = summarize_episodes(anomalies)
print(f"{len(anomalies)} anomalous hours -> {len(episodes)} episodes")
episodes.head(10)

## Validation against injected ground truth
The synthetic generator records every injected event, allowing a recall check (with real data, replace by stakeholder plausibility review).

In [ ]:
truth = ds.ground_truth
night = truth[truth["type"] == "night_episode"]
for _, ev in night.iterrows():
    hit = anomalies[(anomalies["building_id"] == ev["building_id"])
                    & anomalies["timestamp"].between(ev["start"], ev["start"] + pd.Timedelta(hours=int(ev["hours"]))) ]
    print(f"{ev['building_id']} night episode @ {ev['start']}: {'DETECTED' if len(hit) else 'MISSED'} ({len(hit)} hours)")

## Worst episode drill-down

In [ ]:
ep = episodes.iloc[0]
w = clean[(clean["building_id"] == ep["building_id"])
          & clean["timestamp"].between(ep["start"] - pd.Timedelta("2D"), ep["end"] + pd.Timedelta("2D"))]
m = anomalies[(anomalies["building_id"] == ep["building_id"])
              & anomalies["timestamp"].between(ep["start"], ep["end"])]
fig = px.line(w, x="timestamp", y="meter_reading",
              title=f"{ep['building_id']} — {ep['detectors']} (severity {ep['peak_severity']:.0f})")
fig.add_scatter(x=m["timestamp"], y=m["value"], mode="markers", name="anomalous",
                marker=dict(color="red", size=8, symbol="x"))
fig.show()

## KPIs and recommendations (RQ4)

In [ ]:
kpis = compute_kpis(clean, ds.metadata)
kpis

In [ ]:
recs = generate_recommendations(kpis, episodes)
recs[["building_id", "priority", "title", "est_saving_eur_yr", "est_saving_co2_t_yr"]]

## Conclusions
- Night-consumption episodes and operating-hour zero-runs are found with full recall
  on the ground truth; false-positive discussion belongs in the report.
- Recommendations quantify € and CO₂ impact using configurable business parameters
  and are surfaced as cards in the dashboard (`streamlit run dashboard/app.py`).